In [ ]:
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from datasets import Dataset
import evaluate
import torch
import numpy as np
from transformers import DataCollatorForSeq2Seq
from transformers import Trainer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import argparse
import os
import random

In [3]:
from constants import COLLECTIVE_TXT, RESPONSE_TXT, PROPERTY_TXT, CONCLUSION_TXT, MAX_LENGTH
from load_dataset import load_asohmo
from utils import print_trainable_parameters

In [19]:
# Define the arguments
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f' Device: {device}')
parser = argparse.ArgumentParser(description="Train models and generate counter-narratives against hate speech")
parser.add_argument("--generation_strategy", type=str, choices=["zeroshot", "fewshot", "finetuning"])
parser.add_argument("--use_extra_info", type=str, choices=["collective", "premises", "all", ""], default="")
parser.add_argument("--cn_strategy", type=str, default="", choices=["a", "b", "c", ""])
parser.add_argument("--model_name", type=str, default="google/flan-t5-base")
parser.add_argument("--quantized", action="store_true", default=False)
parser.add_argument("--load_from_hub", action="store_true", default=False)
parser.add_argument("--fewshot_examples", type=int, default=5)
parser.add_argument("--short", action="store_true", default=False)


# Define models to test
models = ["meta-llama/Llama-3.2-3B", "mistralai/Mistral-Small-Instruct-2409", "tiiuae/Falcon3-3B-Instruct", "google/flan-t5-base", "google/gemma-2-2b"]

# Simulate the CLI args in Jupyter
test_args = [
    "--generation_strategy", "fewshot",
    "--use_extra_info", "all",
    "--cn_strategy", "a",
    "--model_name", "tiiuae/Falcon3-3B-Instruct",
    # "--quantized", # uncomment to set to true
    "--load_from_hub",
    "--fewshot_examples", "5",
    # "--short",
]
args = parser.parse_args(test_args)


 Device: cuda


In [21]:
model_name = args.model_name
language = "spanish"
dataset = ""
finetuning = args.generation_strategy == "finetuning"
is_causallm = "flan-t5" not in model_name
model_without_user_interface = "tiiuae/falcon" in model_name
extra_info = args.use_extra_info if args.use_extra_info != "" else "no-info"
cn_strategy = args.cn_strategy if args.cn_strategy != "" else "no-strategy"
repository_id = f"{model_name.split('/')[1]}_{language}_{extra_info}_{cn_strategy}"
SHORT = args.short


In [34]:
def add_arg_info(extra_info_sample, tweet):
    if "collective" in extra_info_sample and extra_info_sample["collective"].strip() != "":
        collective = extra_info_sample["collective"]
        tweet += f" | {COLLECTIVE_TXT}{collective}"
    if "property" in extra_info_sample and extra_info_sample["property"].strip() != "":
        propert = extra_info_sample["property"]
        tweet += f" | {PROPERTY_TXT}{propert}"
    if "justification" in extra_info_sample and extra_info_sample["justification"].strip() != "":
        justification = extra_info_sample["justification"]
        tweet += f" | {CONCLUSION_TXT}{justification}"
    if "conclusion" in extra_info_sample and extra_info_sample["conclusion"].strip() != "":
        conclusion = extra_info_sample["conclusion"]
        tweet += f" | {CONCLUSION_TXT}{conclusion}"
    return tweet

In [35]:
def generate_prompt(text, extra_info, fewshot_examples, short=False):
    initial_prompt = "Sos un operario de una ONG experto en generación de contra-narrativas y contra-discurso contra el discurso de odio. Solo hablas Español y eres incapáz de generar texto en otro idioma. Tu tarea es generar una respuesta en Español a un tweet de odio."
    argumentative_prompt = " También se te proporcionará información argumentativa sobre el tweet separada por '|'. Utiliza esta información para elaborar una respuesta. No incluyas esta información en la respuesta."
    final_prompt = " Responde el tweet directamente sin agregar ninguna otra información que no sea la respuesta. El tweet con el mensaje de odio es el siguiente:\n\n"
    has_arg_info = "collective" in extra_info or "property" in extra_info or "justification" in extra_info or "conclusion" in extra_info
    if model_without_user_interface or not is_causallm:
        if short:
                prompt = "Escribe una respuesta en contra del odio para el siguiente tweet de odio: "
        else:
            prompt = initial_prompt
            if has_arg_info:
                prompt += argumentative_prompt
            prompt += final_prompt
        if fewshot_examples:
            #TODO: Move this to a separate function because its repeated in the other case
            counter = 0
            for example in fewshot_examples:
                if counter == args.fewshot_examples:
                    break
                else:
                    counter += 1
                tweet = example["hateSpeech"]
                cn = example["counterSpeech"]
                extra_info_sample = example["extra_info"]
                tweet = add_arg_info(extra_info_sample, tweet)
                prompt += f"'{tweet}'{RESPONSE_TXT}'{cn}'\n"
        text = add_arg_info(extra_info, text)
        prompt += f"'{text}'{RESPONSE_TXT} "
    else:
        prompt = [{"role": "system", "content": initial_prompt}]
        if fewshot_examples:
            counter = 0
            for example in fewshot_examples:
                if counter == args.fewshot_examples:
                    break
                else:
                    counter += 1
                tweet = example["hateSpeech"]
                cn = example["counterSpeech"]
                extra_info_sample = example["extra_info"]
                tweet = add_arg_info(extra_info_sample, tweet)
                prompt.append({"role": "user", "content": tweet})
                prompt.append({"role": "assistant", "content": cn})
        text = add_arg_info(extra_info, text)
        prompt.append({"role": "user", "content": text})
    return prompt

In [36]:
# Load dataset
print("loading asohmo...")
exxtra_info = args.use_extra_info
if args.cn_strategy == "a":
    if args.use_extra_info == "premises":
        exxtra_info = "cn_a"
    else:
        exxtra_info = "cn_a_no_info"
elif args.cn_strategy == "b":
    if args.use_extra_info == "collective":
        exxtra_info = "cn_b"
    else:
        exxtra_info = "cn_b_no_info"
elif args.cn_strategy == "c":
    if args.use_extra_info == "premises":
        exxtra_info = "cn_c"
    else:
        exxtra_info = "cn_c_no_info"
datasetss = load_asohmo(use_extra_info=exxtra_info, finetuning=finetuning)

loading asohmo...
12 - 33 - 7
Counter narratives without the required type of counter-narrative: 4
Non arg examples discarted for not having CN: 52
27 - 101 - 12


In [37]:
test_dataset = datasetss[0]
test_data = Dataset.from_pandas(pd.DataFrame(test_dataset))

In [38]:
train_dataset = datasetss[1]
val_dataset = datasetss[2]
train_data = Dataset.from_pandas(pd.DataFrame(train_dataset + val_dataset))

In [39]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if "Mistral" in model_name or "Mixtral" in model_name or "Qwen" in model_name:
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="left",
        add_eos_token=True,
        add_bos_token=True,
)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

if args.generation_strategy == "finetuning":
    # if args.cn_strategy != "":
    finetuning_name = f"{args.model_name.split('/')[-1]}_{language}_{extra_info}_{cn_strategy}"
    if args.load_from_hub:
        model_name = f"CounterNarratives/{finetuning_name}"
    else:
        model_name = f"pretrained_models/{finetuning_name}"
    print("LOADING MODEL: ", model_name)
    # else:
    #     model_name = f"pretrained_models/{dataset}_{args.model_name.replace('/', '-')}_multi_{args.use_extra_info}_2e-05_8Epochs"


In [40]:
if args.quantized:
    print("Using Quantized model")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    if is_causallm:
        model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, force_download=True)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config)
    if args.generation_strategy == "finetuning":
        model.gradient_checkpointing_enable()
        model = prepare_model_for_kbit_training(model)
        config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
                "lm_head",
            ],
            bias="none",
            lora_dropout=0.05,  # Conventional
            task_type=TaskType.CAUSAL_LM,
        )
        model = get_peft_model(model, config)
        print_trainable_parameters(model)
else:
    if is_causallm:
        model = AutoModelForCausalLM.from_pretrained(model_name)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        model.resize_token_embeddings(len(tokenizer))
    model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [41]:
datasett = test_dataset
if finetuning:
    datasett += train_dataset
    datasett += val_dataset

In [42]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(131072, 3072)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (up_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-06)
    (rotary_emb

In [43]:
def preprocess(sample, padding="max_length", is_testing = False, fewshot_examples = None):
    inputs = generate_prompt(sample["hateSpeech"], sample["extra_info"], fewshot_examples, short=SHORT)
    # ES NECESARIO ESTO?
    if finetuning:
        if is_causallm:
            if model_without_user_interface:
                model_inputs = tokenizer(inputs + sample["counterSpeech"], padding=padding, max_length=MAX_LENGTH, truncation=True)
            else:
                inputs.append({"role": "assistant", "content": sample["counterSpeech"]})
                input_with_chat_template = tokenizer.apply_chat_template(inputs, return_tensors="pt", tokenize=False, add_generation_prompt=False)
                model_inputs = tokenizer(input_with_chat_template, padding=padding, max_length=MAX_LENGTH, truncation=True)
            model_inputs["labels"] = model_inputs["input_ids"].copy()

        else:
            model_inputs = tokenizer(inputs, padding=padding, max_length=MAX_LENGTH, truncation=True)
            labels = tokenizer(sample["counterSpeech"], padding=padding, max_length=MAX_LENGTH, truncation=True)
            if padding == "max_length":
                labels["input_ids"] = [
                    (l if l != tokenizer.pad_token_id else -100) for l in labels["input_ids"]
                ]
            model_inputs["labels"] = labels["input_ids"]
    else:
        if model_without_user_interface or not is_causallm:
            model_inputs = tokenizer(inputs, padding=padding, max_length=MAX_LENGTH, truncation=True, return_tensors="pt")
        else:
            input_with_chat_template = tokenizer.apply_chat_template(inputs, return_tensors="pt", add_generation_prompt=True, tokenize = False)
            model_inputs = tokenizer(input_with_chat_template, padding=padding, max_length=MAX_LENGTH, truncation=True, return_tensors="pt")

    if is_testing:
        test_inputs = {"example": model_inputs, "counterSpeech": sample["counterSpeech"], "number": sample["extra_info"]["number"], "tweet": sample["hateSpeech"]}
        return test_inputs
    return model_inputs

In [44]:
train_data = train_data.map(preprocess)

Map:   0%|          | 0/113 [00:00<?, ? examples/s]

In [ ]:
if finetuning:
    print("Finetuning model..")
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        # Replace -100 in the labels as we can't decode them.
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        # decoded_inputs = tokenizer.batch_decode(inputs, skip_special_tokens=True)

        # Some simple post-processing
        # decoded_preds, decoded_labels, decoded_inputs = postprocess_text(decoded_preds, decoded_labels, decoded_inputs)
        metric = evaluate.load("bertscore")
        # Using rouge score
        result = metric.compute(predictions=decoded_preds, references=decoded_labels)


        # result = {k: round(v * 100, 4) for k, v in result.items()}
        prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
        result["gen_len"] = np.mean(prediction_lens)
        result["prediction"] = decoded_preds
        result["labels"] = decoded_labels
        # result["inputs"] = decoded_inputs
        return result

    # we want to ignore tokenizer pad token in the loss
    label_pad_token_id = -100
    # Data collator
    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=label_pad_token_id,
    )

    optimizer = "adamw_bnb_8bit" if args.quantized else "adamw_torch"
    # Define training args
    training_args = TrainingArguments(
        output_dir=repository_id,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        # predict_with_generate=True,
        # generation_max_length=200,
        # generation_num_beams=4,
        # TODO: turn this on and check if it works
        fp16=False, # Overflows with fp16
        learning_rate=2e-04,
        num_train_epochs=8,
        lr_scheduler_type="cosine",
        # include_inputs_for_metrics=True,
        # logging & evaluation strategies
        # logging_dir=f"{repository_id}/logs",
        # logging_strategy="steps",
        # logging_steps=5,
        optim=optimizer,
        # evaluation_strategy="epoch",
        # save_strategy="epoch",
        # save_total_limit=10,
        # load_best_model_at_end=True,
        # metric_for_best_model="overall_f1",
        # push to hub parameters
        # report_to="tensorboard",
        # push_to_hub=True,
        # hub_strategy="every_save",
        # hub_model_id=repository_id,
    )

    # Create Trainer instance
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_data,
        # eval_dataset=val_data,
        # compute_metrics=compute_metrics,
    )

    model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
    trainer.train()

    if not os.path.exists("pretrained_models"):
        os.makedirs("pretrained_models")
    model.save_pretrained("pretrained_models/" + repository_id)

    # Configurar token de HuggingFace después
    # model.push_to_hub("CounterNarratives/" + repository_id)

else:
    fewshot_examples = train_data if args.generation_strategy == "fewshot" else None
    preprocessed_dataset = []
    for example in test_data:
        preprocessed_dataset.append(preprocess(example, is_testing = True, fewshot_examples = fewshot_examples))

    print("generating...")
    # evaluate_generation(preprocessed_dataset)
    # evaluate_generation(preprocessed_dataset, top_sampling=True)
    # evaluate_generation(preprocessed_dataset, temperature=True)
    # evaluate_generation(preprocessed_dataset, beam_search=True, is_causallm=is_causallm)

generating...


In [54]:
# Seed the random number generator for reproducibility
random.seed(343)

# Select 5 random indices
random_indices = random.sample(range(len(preprocessed_dataset)), k=min(5, len(preprocessed_dataset)))
print(random_indices)

[6, 15, 24, 5, 14]


In [ ]:
print("\nSelected 5 random examples:")
for idx in random_indices:
    sample = preprocessed_dataset[idx]
    print(f"\nTweet: {sample['tweet']}\n")
    print(f"CounterSpeech: {sample['counterSpeech']} \n")
    print("-" * 50)


Selected 5 random examples:

Tweet: a ver quién paga la factura y cómo se reparten el marrón de dejar inmigrantes ilegales sueltos por sus ciudades (y que los votantes no de lo hagan paga r dentro de un año) https://t co/cehrdjssdu

CounterSpeech: ['Pues sería una lastima que el resultado de las elecciones cambie tanto por aceptar a algunos inmigrantes. Hablaría muy mal de la sociedad.\n'] 

--------------------------------------------------

Tweet: españa está siendo invadida por los subsaharianos y aquí don tonto jugando al corro la patata desde la moncloa hashtag caminodevenezuela hashtag señorllevamépronto https://t co/h4xmwqlvw7

CounterSpeech: ['No entiendo a qué te refieres, o qué te preocupa. Ni siquiera propones nada. No tiene mucho sentido lo qué dices.\n'] 

--------------------------------------------------

Tweet: " mientras los manteros atacan a turistas y hay reyertas en el raval ada colau ha visitado el pueblo italiano de riace repoblado dando casas a inmigrantes y ref